# Imports and setup

In [1]:
import pandas as pd
from scipy.stats import norm

from churn.config import LOSS_IF_CHURN, OFFER_COST, SAVE_RATE

# Sample size function

In [2]:
def n_per_group(p1, p2, alpha=0.05, power=0.8):
    za, zb = norm.ppf(1 - alpha / 2), norm.ppf(power)
    return (za + zb) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p1 - p2) ** 2

# Effect sizes and sample sizes

In [3]:
baseline = 0.369                              # churn share among flagged customers (precision on test)
breakeven = OFFER_COST / LOSS_IF_CHURN        # needed absolute reduction

scenarios = {
    "assumed effect (30% saved)": baseline * (1 - SAVE_RATE),
    "half of the assumed effect": baseline * (1 - SAVE_RATE / 2),
    "break-even effect": baseline - breakeven,
}
pd.DataFrame(
    [{"scenario": k, "treatment churn": round(v, 3), "n per group": round(n_per_group(baseline, v))}
     for k, v in scenarios.items()]
)

,scenario,treatment churn,n per group
0,assumed effect (30% saved),0.258,272
1,half of the assumed effect,0.314,1148
2,break-even effect,0.347,7139


# Conclusions

- Baseline churn among flagged customers is about 36.9% (precision on the test set).
- The break-even reduction is c / V = 20 / 890, about 2.25 percentage points: a smaller effect does not cover the cost of the offer.
- Sample size per group (two-sided test of two proportions, alpha 0.05, power 0.8): about 272 to detect the assumed effect (30% saved), about 1148 for half of it, about 7139 for the break-even effect.
- Detecting only the break-even effect would need roughly 14 000 customers in total, more than the whole dataset, so a real test should be planned for a realistic effect size.
- This is a design only: no experiment was run, and the 30% save rate remains an assumption.